# 09_extract_whalen_benchmark

**Objective:** Extract the Whalen et al. (2020) top-1 patent-similarity score for the target US utility grants, used as the external benchmark for competitive density.

**Inputs:** `most_sim.zip` (Whalen `most_sim.json`); `whalen_match_subsample.parquet`.

**Outputs:** `whalen_top1_matched.parquet`.

In [ ]:
# Imports
import json
import time
import zipfile
from pathlib import Path

import pandas as pd

## Paths and load target grant numbers

In [ ]:
# Paths and target grant numbers
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
ZIP_PATH = RAW / "benchmarks" / "most_sim.zip"
INNER    = "most_sim.json"
SUB_PATH = PROC / "whalen_match_subsample.parquet"
OUT_PATH = PROC / "whalen_top1_matched.parquet"

sub = pd.read_parquet(SUB_PATH)
targets = set(sub["grant_num"].astype(str).tolist())
print(f"targets: {len(targets):,} grant numbers")

## Stream JSONL, fast-reject non-target lines, keep top-1 neighbor

In [ ]:
# Stream the Whalen similarity file, keeping the top-1 neighbour for target grants
found = {}
t0 = time.time()
n_lines = 0
with zipfile.ZipFile(ZIP_PATH) as zf:
    with zf.open(INNER) as fh:
        for raw in fh:
            n_lines += 1
            if n_lines % 500_000 == 0:
                rate = n_lines / (time.time() - t0)
                print(f"  {n_lines:>10,}  found={len(found):>6,}  ({rate:,.0f} lines/s)")
            try:
                q1 = raw.index(b'"')
                q2 = raw.index(b'"', q1 + 1)
                key = raw[q1 + 1:q2].decode("ascii")
            except ValueError:
                continue
            if key not in targets:
                continue
            rec = json.loads(raw)
            _, neighbors = rec
            if not neighbors:
                continue
            top1_id, top1_sim = neighbors[0]
            found[key] = (str(top1_id), float(top1_sim))

print(f"\n  done. lines={n_lines:,}  matched={len(found):,}  elapsed={(time.time()-t0)/60:.1f} min")

## Assemble and save results

In [ ]:
# Assemble and save the matched top-1 similarities
df = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in found.items()],
    columns=["grant_num", "whalen_top1_id", "whalen_top1_sim"],
)
df.to_parquet(OUT_PATH)
print(f"saved → {OUT_PATH} ({len(df):,} rows)")